# Stage 2C — synthetic glare training pairs (Flare7K++ on Indian no-glare frames)

**Inputs (Add Input):** `daiyuekun/flare7kpp`, your `vehicle-glare` dataset, `datamon` (yolo11s_best.pt).
**Settings:** GPU T4, Internet on.

**Full run:** `QUICK = False` (Cell 1) → *Save Version → Save & Run All (Commit)*. About 1.5 h.
The output folder `synth/` (zip shards + meta/yolo jsonl, ~8 GB) stays attached to the saved version, so the
decoder-training notebook can add this notebook's output as an input. `stage2c_report.zip` is the small report to download.

Settings v3: flare scale 0.15–1.0 by depth, brightness 0.8–2.0, 5–30% of flare colour kept, lamps only on vehicles or road
between 30% and 72% of frame height, wide haze on 60% of lamps. Quick tests: v2 bright-blob sizes matched real glare.


## 1. Setup

In [ ]:
!pip install -q -U ultralytics
import os, glob, json, random, time, zipfile, shutil
import multiprocessing as mp
from pathlib import Path
import numpy as np, cv2, torch, torch.nn.functional as F
from scipy import ndimage as ndi

QUICK = False         # True: 120 validation samples + all checks (~10 min). False: full set (use Save & Run All)

FL = Path("/kaggle/input/datasets/daiyuekun/flare7kpp/Flare7Kpp")
DS = next(Path(p).parent for p in glob.glob("/kaggle/input/**/data.yaml", recursive=True)
          if (Path(p).parent / "images" / "train").exists())
W_YOLO = glob.glob("/kaggle/input/**/yolo11s_best.pt", recursive=True)[0]
WORK = Path("/kaggle/working"); OUT = WORK / "synth"; OUT.mkdir(exist_ok=True)
REP = WORK / "stage2c_report"; REP.mkdir(exist_ok=True)

W, H = 960, 540              # frame size used everywhere
OW, OH = 480, 272            # size of cached model outputs
PER_BG = {"train": 8, "val": 4, "test": 4}   # synthetic pairs per no-glare background
A_MIN = 0.15                 # attenuation floor
A_EPS = 0.01                 # linear-light offset in A* = (Y(C)+eps)/(Y(I)+eps); stops near-black pixels getting noisy targets
SHARD = 1000                 # samples per zip shard
SEED = 0
cv2.setNumThreads(1)
print("dataset:", DS, "| flares:", FL, "| detector:", W_YOLO, "| QUICK =", QUICK)

## 2. Backgrounds and flare lists

In [ ]:
def has_box(img):
    p = str(img).replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    return os.path.exists(p) and any(l.strip() for l in open(p))

NEG = {s: sorted(p for p in (DS / "images" / s).glob("*.jpg") if not has_box(p)) for s in ["train", "val", "test"]}
BGPATH = {p.name: p for s in NEG for p in NEG[s]}
GLARE_VAL = sorted(p for p in (DS / "images" / "val").glob("*.jpg") if has_box(p))
print("no-glare backgrounds:", {s: len(v) for s, v in NEG.items()}, "| real glare val images:", len(GLARE_VAL))

def pairs(comp_dir, ls_dir):
    return [(str(p), str(Path(ls_dir) / p.name)) for p in sorted(Path(comp_dir).glob("*.png"))]
SCAT = pairs(FL / "Flare7K/Scattering_Flare/Compound_Flare", FL / "Flare7K/Scattering_Flare/Light_Source")
REAL = pairs(FL / "Flare-R/Compound_Flare", FL / "Flare-R/Light_Source")
REFL = [str(p) for p in sorted((FL / "Flare7K/Reflective_Flare").glob("*.png"))]

def split_flares(lst, seed):
    """90% of flare patterns for training; the other 10% for validation and test (never seen in training)"""
    idx = list(range(len(lst))); random.Random(seed).shuffle(idx); k = len(lst) // 10
    return {"train": [lst[i] for i in idx[k:]], "val": [lst[i] for i in idx[:k]], "test": [lst[i] for i in idx[:k]]}
FLARES = {"scat": split_flares(SCAT, 1), "real": split_flares(REAL, 2), "refl": split_flares(REFL, 3)}
for name, lst in [("scattering", SCAT), ("Flare-R", REAL), ("reflective", REFL)]:
    im = cv2.imread(lst[0][0] if isinstance(lst[0], tuple) else lst[0], cv2.IMREAD_UNCHANGED)
    print(f"{name}: {len(lst)} images, e.g. {im.shape} {im.dtype}")

## 3. Frozen models

In [ ]:
from transformers import SegformerForSemanticSegmentation, AutoModelForDepthEstimation
from ultralytics import YOLO

dev = "cuda"
SEGM = SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b2-finetuned-cityscapes-1024-1024").to(dev).eval().half()
DEPM = AutoModelForDepthEstimation.from_pretrained("depth-anything/Depth-Anything-V2-Small-hf").to(dev).eval().half()
GLM = YOLO(W_YOLO)            # our glare detector
COCO = YOLO("yolo11m.pt")     # vehicles in the backgrounds (where to put headlights)

L2I = {v: int(k) for k, v in SEGM.config.id2label.items()}
GROUPS = {"vru": ["person", "rider", "bicycle", "motorcycle"], "vehicle": ["car", "truck", "bus", "train"],
          "drivable": ["road", "sidewalk"], "signal": ["traffic light", "traffic sign", "pole"], "sky": ["sky"],
          "background": ["building", "wall", "fence", "vegetation", "terrain"]}
GID = {g: [L2I[c] for c in cs] for g, cs in GROUPS.items()}
MEAN = torch.tensor([0.485, 0.456, 0.406], device=dev).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=dev).view(1, 3, 1, 1)

def load_rgb(p):
    return cv2.resize(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB), (W, H), interpolation=cv2.INTER_AREA)

def to_batch(imgs, size):
    x = torch.from_numpy(np.stack([cv2.resize(i, size, interpolation=cv2.INTER_AREA) for i in imgs])).to(dev)
    return ((x.permute(0, 3, 1, 2).float() / 255 - MEAN) / STD).half()

@torch.no_grad()
def seg_batch(imgs):
    """RGB uint8 960x540 list -> class map (uint8) and confidence (uint8, x255) at 480x272"""
    logits = SEGM(pixel_values=to_batch(imgs, (960, 544))).logits
    prob = F.interpolate(logits.float(), size=(OH, OW), mode="bilinear", align_corners=False).softmax(1)
    conf, cls = prob.max(1)
    return cls.cpu().numpy().astype(np.uint8), (conf * 255).round().byte().cpu().numpy()

@torch.no_grad()
def depth_batch(imgs, egos):
    """relative inverse depth (1 = near), normalised per frame over pixels outside the own-vehicle mask"""
    d = DEPM(pixel_values=to_batch(imgs, (924, 518))).predicted_depth
    d = F.interpolate(d[:, None].float(), size=(OH, OW), mode="bilinear", align_corners=False)[:, 0].cpu().numpy()
    out = []
    for di, e in zip(d, egos):
        valid = ~e if e is not None else np.ones(di.shape, bool)
        lo, hi = np.percentile(di[valid], [2, 98])
        out.append(np.clip((di - lo) / (hi - lo + 1e-6), 0, 1))
    return np.stack(out)
print("models loaded")

## 4. Own-vehicle masks

In [ ]:
# Own-vehicle mask per video source (dashboard, handlebar, bonnet), from train frames only:
#  rule 1: lower-frame pixels SegFormer calls vehicle/rider in >= 60% of frames
#  rule 2: lower-frame pixels that hardly change over time
#  rule 3 (fallback when rules 1-2 find < 2%): bottom-30% pixels that are dark in the median frame (e.g. the v6 bonnet)
# each mask is smoothed and must touch the bottom edge of the frame
from IPython.display import Image as ShowImage, display
def src_of(name): return name.split("_")[0]

def clean_mask(m, pad=10):
    """smooth the mask (padding first so the frame border is not eroded) and keep parts touching the bottom edge"""
    p = np.pad(m, pad, mode="edge")
    p = ndi.binary_closing(ndi.binary_opening(p, iterations=1), iterations=4)[pad:-pad, pad:-pad]
    lab, n = ndi.label(p)
    keep = [i for i in range(1, n + 1) if (lab[-6:] == i).any()]
    return ndi.binary_fill_holes(np.isin(lab, keep))

EGO, EGO_S, EGO_DIAG = {}, {}, {}
yrel = (np.arange(OH) / OH)[:, None]
for src in ["v1", "v5", "v6"]:
    files = sorted((DS / "images" / "train").glob(f"{src}_*.jpg"))
    files = files[::max(1, len(files) // 240)]
    freq = np.zeros((OH, OW)); grays = []
    for i in range(0, len(files), 16):
        imgs = [load_rgb(f) for f in files[i:i + 16]]
        cls, _ = seg_batch(imgs)
        freq += np.isin(cls, GID["vehicle"] + GID["vru"]).sum(0)
        grays += [cv2.resize(cv2.cvtColor(im, cv2.COLOR_RGB2GRAY), (OW, OH)).astype(np.float32) / 255 for im in imgs]
    freq /= len(files); grays = np.stack(grays); std, med = grays.std(0), np.median(grays, 0)
    m = clean_mask(((freq >= 0.6) | (std < 0.02)) & (yrel > 0.45))
    rule = "vehicle-label / static"
    if m.mean() < 0.02:
        m = clean_mask((med < 0.08) & (yrel > 0.70))
        rule = "dark in median frame (fallback)"
    EGO_DIAG[src] = (freq, med)
    EGO_S[src] = m
    EGO[src] = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
    print(f"{src}: {len(files)} frames | rule: {rule} | own-vehicle mask = {EGO[src].mean():.1%} of the frame")

def ego_full(name): return EGO.get(src_of(name))
def ego_small(name): return EGO_S.get(src_of(name))

import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 3, figsize=(18, 9.5))
for r, src in enumerate(["v1", "v5", "v6"]):
    im = load_rgb(sorted((DS / "images" / "train").glob(f"{src}_*.jpg"))[50]).astype(np.float32) / 255
    im[EGO[src]] = 0.5 * im[EGO[src]] + 0.5 * np.array([1.0, 0.0, 1.0])
    freq, med = EGO_DIAG[src]
    axes[r, 0].imshow(im); axes[r, 0].set_title(f"{src}: own-vehicle mask (magenta)", loc="left", fontsize=10)
    axes[r, 1].imshow(freq, vmin=0, vmax=1, cmap="viridis"); axes[r, 1].set_title("share of frames labelled vehicle/rider", loc="left", fontsize=10)
    axes[r, 2].imshow(np.clip(med * 2.5, 0, 1), cmap="gray"); axes[r, 2].set_title("median frame (brightened)", loc="left", fontsize=10)
    for a in axes[r]: a.axis("off")
plt.tight_layout(); plt.savefig(REP / "ego_masks.png", dpi=80); plt.close()
display(ShowImage(str(REP / "ego_masks.png")))

## 5. Analyse backgrounds (vehicles, road, depth)

In [ ]:
# For every no-glare background: vehicles (COCO YOLO11m), drivable area (SegFormer) and depth -> where headlights can go
use = {"val": NEG["val"][:30]} if QUICK else NEG
BG = {}
for split, files in use.items():
    for i in range(0, len(files), 16):
        chunk = files[i:i + 16]
        imgs = [load_rgb(f) for f in chunk]
        cls, _ = seg_batch(imgs)
        dep = depth_batch(imgs, [ego_small(f.name) for f in chunk])
        res = COCO.predict([str(f) for f in chunk], imgsz=960, conf=0.35, classes=[2, 3, 5, 7], verbose=False)
        for f, c, d, r in zip(chunk, cls, dep, res):
            ego = ego_full(f.name)
            sy, sx = H / r.orig_shape[0], W / r.orig_shape[1]
            veh = []
            for (x1, y1, x2, y2), k in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int)):
                x1, x2, y1, y2 = x1 * sx, x2 * sx, y1 * sy, y2 * sy
                cx, cy = int((x1 + x2) / 2), int(min((y1 + y2) / 2, H - 1))
                if ego is not None and ego[cy, min(cx, W - 1)]:
                    continue
                veh.append([float(x1), float(y1), float(x2), float(y2), int(k)])
            drv = np.isin(c, GID["drivable"])
            if ego_small(f.name) is not None:
                drv &= ~ego_small(f.name)
            BG[f.name] = {"split": split, "veh": veh, "drv": drv, "dep": d.astype(np.float16)}
print(len(BG), "backgrounds analysed |", sum(len(b["veh"]) > 0 for b in BG.values()), "contain at least one vehicle")

## 6. How one sample is made

In [ ]:
# One training sample = glare frame I at t, t-1, t-2 + clean target C + target attenuation A* (t and t-1) + flare mask M
# v3 settings (quick test 1: glare 3-4x too small, too colourful, in impossible places; test 2: sizes match real glare,
# a few lamps still pink or placed in the sky / on the door -> less colour, lamps only on vehicles or the road)
S_FAR, S_NEAR, S_EXP = 0.15, 1.00, 1.0   # flare scale for far (depth 0) and near (depth 1) lamps
GAIN = (0.8, 2.0)                        # flare brightness range
DESAT = (0.05, 0.3)                      # keep 5-30% of the flare's own colour (Flare-R flares are pink/purple)
BAND = (0.30, 0.72)                      # headlights only appear between these heights (fraction of frame height)
P_BLOOM = 0.6                            # share of lamps with an extra wide haze (veiling glare through the windshield)
YY, XX = np.mgrid[0:H, 0:W].astype(np.float32)
def read_png(path):
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    im = im.astype(np.float32) / (65535.0 if im.dtype == np.uint16 else 255.0)
    if im.ndim == 2:
        im = np.repeat(im[..., None], 3, 2)
    return cv2.cvtColor(np.ascontiguousarray(im[..., :3]), cv2.COLOR_BGR2RGB)

def lamp_centre(L, Fc):
    """centre of the light source inside the flare image (brightness-weighted centroid of the brightest part)"""
    y = L.mean(2) if L.max() > 0.05 else Fc.mean(2)
    m = y >= 0.5 * y.max()
    ys, xs = np.nonzero(m); w = y[m]
    return float((xs * w).sum() / w.sum()), float((ys * w).sum() / w.sum())

def place(img, centre, target, scale, angle):
    """rotate+scale the flare image about its lamp centre and move the lamp centre to `target` in a W x H frame"""
    M = cv2.getRotationMatrix2D(centre, angle, scale)
    M[:, 2] += np.array(target) - np.array(centre)
    return cv2.warpAffine(img, M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)

def lum(x): return 0.2126 * x[..., 0] + 0.7152 * x[..., 1] + 0.0722 * x[..., 2]

def depth_at(bg, x, y):
    return float(bg["dep"][min(int(y * OH / H), OH - 1), min(int(x * OW / W), OW - 1)])

def sample_lamps(rng, bg, ego):
    """headlight positions: on detected vehicles (75%) or on the road area (25%); 1-2 light sources per frame"""
    events = []
    for _ in range(rng.choice([1, 2], p=[0.65, 0.35])):
        mode = rng.choice(["vehicle", "road"], p=[0.75, 0.25])
        if mode == "vehicle" and bg["veh"]:
            wts = np.array([np.sqrt(max((v[2] - v[0]) * (v[3] - v[1]), 1.0)) for v in bg["veh"]])   # prefer nearer (bigger) vehicles
            x1, y1, x2, y2, k = bg["veh"][rng.choice(len(bg["veh"]), p=wts / wts.sum())]
            w, h, cx = x2 - x1, y2 - y1, (x1 + x2) / 2
            if k == 3:                                            # motorcycle: one lamp
                pts = [(cx, y2 - 0.45 * h)]
            else:                                                 # car / bus / truck: two lamps
                sp = rng.uniform(0.30, 0.375) * w
                pts = [(cx - sp, y2 - 0.25 * h), (cx + sp, y2 - 0.25 * h)]
        else:
            mode = "road" if mode == "vehicle" else mode
            ys, xs = np.nonzero(bg["drv"])
            ok = (ys >= BAND[0] * OH) & (ys <= BAND[1] * OH)
            if mode == "road" and ok.any():
                i = rng.choice(np.flatnonzero(ok))
                cx, cy = (xs[i] + 0.5) * W / OW, (ys[i] + 0.5) * H / OH
            else:
                mode = "random"; cx, cy = rng.uniform(0.05, 0.95) * W, rng.uniform(BAND[0], BAND[1]) * H
            d = depth_at(bg, cx, cy)
            sp = (6 + 60 * d) * rng.uniform(0.8, 1.2)              # lamp spacing grows as the vehicle gets nearer
            pts = [(cx - sp, cy), (cx + sp, cy)] if (mode == "road" and rng.random() < 0.7) else [(cx, cy)]
        pts = [(x, y) for x, y in pts if 0 <= x < W and BAND[0] * H <= y <= BAND[1] * H and (ego is None or not ego[int(y), int(x)])]
        if pts:
            events.append((mode, pts))
    if not events:
        events = [("random", [(rng.uniform(0.1, 0.9) * W, rng.uniform(BAND[0], BAND[1]) * H)])]
    return events

def enc_jpg(x01, q):
    return cv2.imencode(".jpg", cv2.cvtColor((np.clip(x01, 0, 1) * 255).round().astype(np.uint8), cv2.COLOR_RGB2BGR),
                        [cv2.IMWRITE_JPEG_QUALITY, int(q)])[1].tobytes()

def enc_png(u8): return cv2.imencode(".png", u8)[1].tobytes()

def make_sample(spec):
    sid, bgname, split, seed = spec
    rng = np.random.default_rng(seed)
    bg, ego = BG[bgname], ego_full(bgname)
    g = rng.uniform(1.8, 2.2)                                     # gamma: sRGB <-> linear
    Bl = (load_rgb(BGPATH[bgname]).astype(np.float32) / 255) ** g
    Bl = np.clip(Bl + rng.normal(0, rng.uniform(0, 0.004), Bl.shape).astype(np.float32), 0, 1)

    # flares: one flare pattern per light source (both lamps of a car share it)
    lamps = []
    for mode, pts in sample_lamps(rng, bg, ego):
        kind = "scat" if rng.random() < 0.65 else "real"
        fpath, lpath = FLARES[kind][split][rng.integers(len(FLARES[kind][split]))]
        Fc, L = read_png(fpath), read_png(lpath)
        if L.shape != Fc.shape:
            L = cv2.resize(L, (Fc.shape[1], Fc.shape[0]))
        refl = None
        if rng.random() < 0.2:
            refl = FLARES["refl"][split][rng.integers(len(FLARES["refl"][split]))]
            R = cv2.resize(read_png(refl), (Fc.shape[1], Fc.shape[0]))
            Fc = np.clip(Fc + R * rng.uniform(0.5, 1.0), 0, 1)
        if rng.random() < 0.5:                                    # slight optical blur
            s = rng.uniform(0.5, 2.0); Fc = cv2.GaussianBlur(Fc, (0, 0), s); L = cv2.GaussianBlur(L, (0, 0), s)
        sat = rng.uniform(*DESAT)                                 # headlights are white/yellow, not pink
        Fc = Fc.mean(2, keepdims=True) + sat * (Fc - Fc.mean(2, keepdims=True))
        L = L.mean(2, keepdims=True) + sat * (L - L.mean(2, keepdims=True))
        tint = np.array([1.0, 0.88, 0.70] if rng.random() < 0.5 else [0.92, 0.97, 1.0], np.float32)  # halogen / LED
        gain = tint * rng.normal(1, 0.05, 3).astype(np.float32) * rng.uniform(*GAIN)
        Fl, Ll = (Fc ** g) * gain, (L ** g) * gain
        centre = lamp_centre(L, Fc)
        k0 = 1440.0 / Fc.shape[1]                                 # normalise every flare image to 1440 px wide
        angle = rng.uniform(0, 360)
        for (x, y) in pts:
            d = depth_at(bg, x, y)
            scale = k0 * (S_FAR + (S_NEAR - S_FAR) * d ** S_EXP) * rng.uniform(0.8, 1.25)
            bloom = [round(float(rng.uniform(0.1, 0.5)), 3), round(float((25 + 110 * d) * rng.uniform(0.7, 1.4)), 1)] \
                if rng.random() < P_BLOOM else None             # [amplitude, radius in px] of the wide haze
            lamps.append({"x": x, "y": y, "depth": d, "scale": scale, "mode": mode, "kind": kind,
                          "flare": Path(fpath).name, "refl": Path(refl).name if refl else None, "sat": round(float(sat), 3),
                          "bloom": bloom, "Fl": Fl, "Ll": Ll, "centre": centre, "angle": angle, "tint": tint})

    # pseudo-video: the car drives forward (background zooms about the vanishing point), lamps approach and grow
    vp = (W / 2 + rng.uniform(-60, 60), H * rng.uniform(0.40, 0.50))
    dz, speed, ds = rng.uniform(0.005, 0.02), rng.uniform(1.0, 3.0), rng.uniform(0.02, 0.08)
    files, out = {}, {}
    q = rng.integers(75, 96)
    for k in (0, 1, 2):                                           # k frames before t
        Bk = Bl
        if k:
            Bk = cv2.warpAffine(Bl, cv2.getRotationMatrix2D(vp, 0, 1 - k * dz), (W, H), borderMode=cv2.BORDER_REFLECT)
            if ego is not None:
                Bk[ego] = Bl[ego]                                 # own vehicle moves with the camera
        Fs, Ls = np.zeros_like(Bl), np.zeros_like(Bl)
        for lp in lamps:
            f = 1 - k * dz * speed
            pos = (vp[0] + (lp["x"] - vp[0]) * f, vp[1] + (lp["y"] - vp[1]) * f)
            sc = lp["scale"] * (1 - k * ds)
            Fs += place(lp["Fl"], lp["centre"], pos, sc, lp["angle"])
            Ls += place(lp["Ll"], lp["centre"], pos, sc, lp["angle"])
            if lp["bloom"]:                                       # haze is glare: in I, not in the clean target C
                amp, rad = lp["bloom"][0], lp["bloom"][1] * (1 - k * ds)
                Fs += (amp * np.exp(-((XX - pos[0]) ** 2 + (YY - pos[1]) ** 2) / (2 * rad ** 2)))[..., None] * lp["tint"]
        I = np.clip(Bk + Fs, 0, 1)
        files[f"I{k}.jpg"] = enc_jpg(I ** (1 / g), q)
        if k < 2:
            C = np.clip(Bk + Ls, 0, 1)
            A = np.clip((lum(C) + A_EPS) / (lum(I) + A_EPS), A_MIN, 1)   # A_EPS: ignore flare light too faint to see
            files[f"A{k}.png"] = enc_png((A * 255).round().astype(np.uint8))
            if k == 0:
                files["C.jpg"] = enc_jpg(C ** (1 / g), q)
                M = (lum(I ** (1 / g)) - lum(C ** (1 / g))) > 0.02
                files["M.png"] = enc_png(M.astype(np.uint8) * 255)
                out["glare_frac"] = float(M.mean())
    meta = {"id": sid, "bg": bgname, "split": split, "src": src_of(bgname), "seed": int(seed), "gamma": round(g, 3),
            "jpeg_q": int(q), "motion": {"vp": [round(vp[0], 1), round(vp[1], 1)], "dz": round(dz, 4), "speed": round(speed, 3), "ds": round(ds, 4)},
            "lamps": [{k: (round(v, 3) if isinstance(v, float) else v) for k, v in lp.items() if k not in ("Fl", "Ll", "centre", "angle", "tint")}
                      for lp in lamps], **out}
    return sid, files, meta

## 7. Generate the pairs

In [ ]:
# Generate the pairs on 4 CPU workers and write them into zip shards (synth/<split>_<k>.zip) + one meta file per split
SPLIT_NO = {"train": 0, "val": 1, "test": 2}
def specs_for(split):
    names = sorted(n for n, b in BG.items() if b["split"] == split)
    out = []
    for n in names:
        for j in range(PER_BG[split]):
            i = len(out)
            out.append((f"{split}_{i:05d}", n, split, SEED * 10**7 + SPLIT_NO[split] * 10**6 + i))
    return out

t0 = time.time()
for split in sorted({b["split"] for b in BG.values()}):
    specs = specs_for(split)
    zf, meta_f = None, open(OUT / f"{split}_meta.jsonl", "w")
    with mp.get_context("fork").Pool(4) as pool:
        for i, (sid, files, meta) in enumerate(pool.imap(make_sample, specs, chunksize=4)):
            if i % SHARD == 0:
                if zf: zf.close()
                zf = zipfile.ZipFile(OUT / f"{split}_{i // SHARD:02d}.zip", "w", zipfile.ZIP_STORED)
            for fn, b in files.items():
                zf.writestr(f"{sid}/{fn}", b)
            meta["shard"] = f"{split}_{i // SHARD:02d}.zip"
            meta_f.write(json.dumps(meta) + "\n")
            if i % 500 == 0:
                print(f"{split}: {i}/{len(specs)}  ({time.time() - t0:.0f} s)")
    zf.close(); meta_f.close()
    print(f"{split}: {len(specs)} samples done ({time.time() - t0:.0f} s)")

## 8. Cache model outputs + realism checks

In [ ]:
# Run the frozen models on every synthetic glare frame I0 (as at inference time) and cache their outputs:
# synth/<split>_cache_<k>.zip with <id>/seg.png, <id>/segconf.png, <id>/depth.png; YOLO11s boxes in <split>_yolo.jsonl
def decode(b): return cv2.cvtColor(cv2.imdecode(np.frombuffer(b, np.uint8), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)

def inside(pt, box, f=1.25):
    cx, cy, w, h = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2, (box[2] - box[0]) * f, (box[3] - box[1]) * f
    return abs(pt[0] - cx) <= w / 2 and abs(pt[1] - cy) <= h / 2

check = {"lamps": 0, "lamps_detected": 0, "by_depth": {"far": [0, 0], "mid": [0, 0], "near": [0, 0]}, "images": 0, "images_hit": 0}
t0 = time.time()
for split in sorted({b["split"] for b in BG.values()}):
    metas = [json.loads(l) for l in open(OUT / f"{split}_meta.jsonl")]
    yolo_f = open(OUT / f"{split}_yolo.jsonl", "w")
    zips, czf = {}, None
    for i in range(0, len(metas), 16):
        batch = metas[i:i + 16]
        for m in batch:
            if m["shard"] not in zips: zips[m["shard"]] = zipfile.ZipFile(OUT / m["shard"])
        imgs = [decode(zips[m["shard"]].read(f"{m['id']}/I0.jpg")) for m in batch]
        cls, conf = seg_batch(imgs)
        dep = depth_batch(imgs, [ego_small(m["bg"]) for m in batch])
        res = GLM.predict([cv2.cvtColor(im, cv2.COLOR_RGB2BGR) for im in imgs], imgsz=960, conf=0.05, verbose=False)
        for j, (m, c, cf, d, r) in enumerate(zip(batch, cls, conf, dep, res)):
            k = i + j
            if k % SHARD == 0:
                if czf: czf.close()
                czf = zipfile.ZipFile(OUT / f"{split}_cache_{k // SHARD:02d}.zip", "w", zipfile.ZIP_STORED)
            czf.writestr(f"{m['id']}/seg.png", enc_png(c)); czf.writestr(f"{m['id']}/segconf.png", enc_png(cf))
            czf.writestr(f"{m['id']}/depth.png", enc_png((d * 65535).round().astype(np.uint16)))
            boxes = [[round(float(v), 1) for v in b] + [round(float(s), 3)]
                     for b, s in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy())]
            yolo_f.write(json.dumps({"id": m["id"], "cache": f"{split}_cache_{k // SHARD:02d}.zip", "boxes": boxes}) + "\n")
            strong = [b for b in boxes if b[4] >= 0.25]
            check["images"] += 1; check["images_hit"] += bool(strong)
            for lp in m["lamps"]:
                hit = any(inside((lp["x"], lp["y"]), b) for b in strong)
                band = "far" if lp["depth"] < 0.33 else ("mid" if lp["depth"] < 0.66 else "near")
                check["lamps"] += 1; check["lamps_detected"] += hit
                check["by_depth"][band][0] += 1; check["by_depth"][band][1] += hit
        if i % 800 == 0:
            print(f"{split} cache: {i}/{len(metas)}  ({time.time() - t0:.0f} s)")
    czf.close(); yolo_f.close()
    for z in zips.values(): z.close()

# Check 1: does YOLO11s see the synthetic glare like real glare?  (real images in batches of 16)
real_hit = []
for i in range(0, len(GLARE_VAL), 16):
    res = GLM.predict([str(p) for p in GLARE_VAL[i:i + 16]], imgsz=960, conf=0.25, verbose=False)
    real_hit += [len(r.boxes) > 0 for r in res]
real_hit = float(np.mean(real_hit))
summary = {
    "QUICK": QUICK, "samples": check["images"],
    "yolo_synthetic_images_with_glare_box": round(check["images_hit"] / check["images"], 3),
    "yolo_real_val_glare_images_with_glare_box": round(real_hit, 3),
    "yolo_synthetic_lamp_recall": round(check["lamps_detected"] / max(check["lamps"], 1), 3),
    "yolo_lamp_recall_by_depth": {k: (round(v[1] / v[0], 3) if v[0] else None, v[0]) for k, v in check["by_depth"].items()},
    "rule": "lamp recall >= 0.70 -> train with real YOLO11s boxes; otherwise use simulated boxes",
}

# Check 2: size of saturated blobs (pixels >= 230 grey) per image, synthetic vs real glare (validation split)
def blob_areas(img_rgb):
    m = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY) >= 230
    lab, n = ndi.label(m)
    return np.bincount(lab.ravel())[1:] if n else np.array([0])
syn, rl = [], []
metas = [json.loads(l) for l in open(OUT / "val_meta.jsonl")][:300]
zips = {}
for m in metas:
    if m["shard"] not in zips: zips[m["shard"]] = zipfile.ZipFile(OUT / m["shard"])
    syn.append(blob_areas(decode(zips[m["shard"]].read(f"{m['id']}/I0.jpg"))).max())
for p in GLARE_VAL[:300]:
    rl.append(blob_areas(load_rgb(p)).max())
summary["largest_bright_blob_px_synthetic_p25_p50_p75_p90"] = [int(v) for v in np.percentile(syn, [25, 50, 75, 90])]
summary["largest_bright_blob_px_real_p25_p50_p75_p90"] = [int(v) for v in np.percentile(rl, [25, 50, 75, 90])]
summary["glare_area_share_of_frame_median"] = round(float(np.median([m["glare_frac"] for m in metas])), 4)
summary["minutes"] = round((time.time() - t0) / 60, 1)
json.dump(summary, open(REP / "stage2c_summary.json", "w"), indent=1)
print(json.dumps(summary, indent=1))

## 9. Preview + report zip

In [ ]:
# Preview: 8 synthetic samples next to 4 real glare frames, then a small zip to download
import matplotlib.patches as mpatches
metas = [json.loads(l) for l in open(OUT / "val_meta.jsonl")]
yolo = {json.loads(l)["id"]: json.loads(l)["boxes"] for l in open(OUT / "val_yolo.jsonl")}
pick = metas[::max(1, len(metas) // 8)][:8]
zips = {}
fig, axes = plt.subplots(len(pick), 4, figsize=(20, 2.9 * len(pick)))
for r, m in enumerate(pick):
    if m["shard"] not in zips: zips[m["shard"]] = zipfile.ZipFile(OUT / m["shard"])
    z = zips[m["shard"]]
    I0, C = decode(z.read(f"{m['id']}/I0.jpg")), decode(z.read(f"{m['id']}/C.jpg"))
    A = cv2.imdecode(np.frombuffer(z.read(f"{m['id']}/A0.png"), np.uint8), cv2.IMREAD_GRAYSCALE) / 255
    I2 = decode(z.read(f"{m['id']}/I2.jpg"))
    axes[r, 0].imshow(I0)
    for b in yolo[m["id"]]:
        if b[4] >= 0.25:
            axes[r, 0].add_patch(mpatches.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec="yellow", lw=1.2))
    for lp in m["lamps"]:
        axes[r, 0].plot(lp["x"], lp["y"], "c+", ms=8)
    axes[r, 0].set_title(f"{m['id']} glare input I (cyan + = lamp, yellow = YOLO11s)", fontsize=8, loc="left")
    axes[r, 1].imshow(C); axes[r, 1].set_title("clean target C (lamp kept)", fontsize=8, loc="left")
    axes[r, 2].imshow(A, cmap="gray", vmin=0, vmax=1); axes[r, 2].set_title(f"target attenuation A* (min {A.min():.2f})", fontsize=8, loc="left")
    axes[r, 3].imshow(I2); axes[r, 3].set_title("frame t-2 (pseudo-video)", fontsize=8, loc="left")
    for a in axes[r]: a.axis("off")
plt.tight_layout(); plt.savefig(REP / "synthetic_examples.png", dpi=80); plt.close()

fig, axes = plt.subplots(1, 4, figsize=(20, 2.9))
for a, p in zip(axes, GLARE_VAL[::max(1, len(GLARE_VAL) // 4)][:4]):
    a.imshow(load_rgb(p)); a.set_title(f"REAL {p.name}", fontsize=8, loc="left"); a.axis("off")
plt.tight_layout(); plt.savefig(REP / "real_examples.png", dpi=80); plt.close()

from IPython.display import FileLink, display
shutil.make_archive(str(WORK / REP.name), "zip", root_dir=WORK, base_dir=REP.name)
print("synth folder:", round(sum(f.stat().st_size for f in OUT.glob("*")) / 1e9, 2), "GB")
display(FileLink(REP.name + ".zip"))